# Data Cleaning and Validation
## 1. Importing the data

In [1]:
import pandas as pd

plant1_weather = pd.read_csv('C:/Users/acer/programming/jupyter/2026/solar_power_generation_data/data/Plant_1_Weather_Sensor_Data (2).csv', sep=",")

print(f"Plant 1 Weather Sensor: {plant1_weather.shape}")


plant1_weather.head()

Plant 1 Weather Sensor: (3182, 6)


,DATE_TIME,PLANT_ID,SOURCE_KEY,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION
0,2020-05-15 00:00:00,4135001,HmiyD2TTLFNqkNe,25.184316,22.857507,0.0
1,2020-05-15 00:15:00,4135001,HmiyD2TTLFNqkNe,25.084589,22.761668,0.0
2,2020-05-15 00:30:00,4135001,HmiyD2TTLFNqkNe,24.935753,22.592306,0.0
3,2020-05-15 00:45:00,4135001,HmiyD2TTLFNqkNe,24.846130,22.360852,0.0
4,2020-05-15 01:00:00,4135001,HmiyD2TTLFNqkNe,24.621525,22.165423,0.0


# 2. Understanding the data
## 2.1. Looking at the data

In [2]:
print(plant1_weather.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3182 entries, 0 to 3181
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   DATE_TIME            3182 non-null   object 
 1   PLANT_ID             3182 non-null   int64  
 2   SOURCE_KEY           3182 non-null   object 
 3   AMBIENT_TEMPERATURE  3182 non-null   float64
 4   MODULE_TEMPERATURE   3182 non-null   float64
 5   IRRADIATION          3182 non-null   float64
dtypes: float64(3), int64(1), object(2)
memory usage: 149.3+ KB
None


In [3]:
print(plant1_weather.describe())

        PLANT_ID  AMBIENT_TEMPERATURE  MODULE_TEMPERATURE  IRRADIATION
count     3182.0          3182.000000         3182.000000  3182.000000
mean   4135001.0            25.531606           31.091015     0.228313
std          0.0             3.354856           12.261222     0.300836
min    4135001.0            20.398505           18.140415     0.000000
25%    4135001.0            22.705182           21.090553     0.000000
50%    4135001.0            24.613814           24.618060     0.024653
75%    4135001.0            27.920532           41.307840     0.449588
max    4135001.0            35.252486           65.545714     1.221652


## 2.2 Defining Column Constraints and Data Types

In [4]:
# ==========================================
# STEP 2: DEFINE CONSTRAINTS FOR EACH COLUMN
# ==========================================

CONSTRAINTS = {
    'DATE_TIME': {
        'description': 'Timestamp of measurement',
        'expected_type': 'datetime',
        'format_variants': ['%Y-%m-%d %H:%M:%S'],
        'range': ('2020-05-15 00:00:00', '2020-06-17 23:45:00'),
        'nullable': False
    },
    'PLANT_ID': {
        'description': 'Solar plant identifier',
        'expected_type': 'integer',
        'allowed_values': [4135001],
        'nullable': False
    },
    'SOURCE_KEY': {
        'description': 'Weather sensor identifier',
        'expected_type': 'string',
        'nullable': False
    },
    'AMBIENT_TEMPERATURE': {
        'description': 'Ambient temperature (°C)',
        'expected_type': 'float',
        'min': -50.0,
        'max': 60.0,
        'nullable': False,
        'decimal_separator': '.'
    },
    'MODULE_TEMPERATURE': {
        'description': 'Solar module temperature (°C)',
        'expected_type': 'float',
        'min': -50.0,
        'max': 85.0,
        'nullable': False,
        'decimal_separator': '.'
    },
    'IRRADIATION': {
        'description': 'Solar irradiance (kW/m²)',
        'expected_type': 'float',
        'min': 0.0,
        'max': 1.5,
        'nullable': False,
        'decimal_separator': '.'
    }
}

print("=== CONSTRAINTS DEFINED ===")
for col, rules in CONSTRAINTS.items():
    print(f"\n{col}: {rules['description']}")
    for rule, value in rules.items():
        if rule != 'description':
            print(f"  - {rule}: {value}")

=== CONSTRAINTS DEFINED ===

DATE_TIME: Timestamp of measurement
  - expected_type: datetime
  - format_variants: ['%Y-%m-%d %H:%M:%S']
  - range: ('2020-05-15 00:00:00', '2020-06-17 23:45:00')
  - nullable: False

PLANT_ID: Solar plant identifier
  - expected_type: integer
  - allowed_values: [4135001]
  - nullable: False

SOURCE_KEY: Weather sensor identifier
  - expected_type: string
  - nullable: False

AMBIENT_TEMPERATURE: Ambient temperature (°C)
  - expected_type: float
  - min: -50.0
  - max: 60.0
  - nullable: False
  - decimal_separator: .

MODULE_TEMPERATURE: Solar module temperature (°C)
  - expected_type: float
  - min: -50.0
  - max: 85.0
  - nullable: False
  - decimal_separator: .

IRRADIATION: Solar irradiance (kW/m²)
  - expected_type: float
  - min: 0.0
  - max: 1.5
  - nullable: False
  - decimal_separator: .


## 3. Data Cleaning
#### 3.1. Check for missing values

In [5]:
# Check for missing values
missing_values = plant1_weather.isnull().sum()
print("Missing values per column:")
print(missing_values)
print("\nPercentage missing:")
print((missing_values / len(plant1_weather)) * 100)

Missing values per column:
DATE_TIME              0
PLANT_ID               0
SOURCE_KEY             0
AMBIENT_TEMPERATURE    0
MODULE_TEMPERATURE     0
IRRADIATION            0
dtype: int64

Percentage missing:
DATE_TIME              0.0
PLANT_ID               0.0
SOURCE_KEY             0.0
AMBIENT_TEMPERATURE    0.0
MODULE_TEMPERATURE     0.0
IRRADIATION            0.0
dtype: float64


#### 3.2. Check for duplicate values


In [6]:
# Check for duplicate rows
duplicate_rows = plant1_weather.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")

if duplicate_rows > 0:
    print("\nDuplicate rows (first 5):")
    print(plant1_weather[plant1_weather.duplicated(keep=False)].head())

Number of duplicate rows: 0


#### 3.3. Check for outliers


In [7]:
# Check for outliers using IQR method
import numpy as np

numeric_columns = ['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']

for col in numeric_columns:
    Q1 = plant1_weather[col].quantile(0.25)
    Q3 = plant1_weather[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = plant1_weather[(plant1_weather[col] < lower_bound) | (plant1_weather[col] > upper_bound)]
    print(f"\n{col}:")
    print(f"  Lower bound: {lower_bound:.2f}")
    print(f"  Upper bound: {upper_bound:.2f}")
    print(f"  Number of outliers: {len(outliers)}")
    if len(outliers) > 0:
        print(f"  Outlier range: {outliers[col].min():.2f} to {outliers[col].max():.2f}")


AMBIENT_TEMPERATURE:
  Lower bound: 14.88
  Upper bound: 35.74
  Number of outliers: 0

MODULE_TEMPERATURE:
  Lower bound: -9.24
  Upper bound: 71.63
  Number of outliers: 0

IRRADIATION:
  Lower bound: -0.67
  Upper bound: 1.12
  Number of outliers: 2
  Outlier range: 1.15 to 1.22


#### 3.4. Check for consistency in data


In [8]:
# Check for data consistency
print("=== DATA CONSISTENCY CHECKS ===")

# Check if MODULE_TEMPERATURE is always >= AMBIENT_TEMPERATURE (should be true as modules heat up in sun)
module_colder = plant1_weather[plant1_weather['MODULE_TEMPERATURE'] < plant1_weather['AMBIENT_TEMPERATURE']]
print(f"\nMODULE_TEMPERATURE < AMBIENT_TEMPERATURE: {len(module_colder)} records")
if len(module_colder) > 0:
    print("Sample records where module is colder than ambient:")
    print(module_colder[['DATE_TIME', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE']].head())

# Check for impossible irradiation values during night hours
plant1_weather['DATE_TIME_DT'] = pd.to_datetime(plant1_weather['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')
plant1_weather['HOUR'] = plant1_weather['DATE_TIME_DT'].dt.hour
night_irradiation = plant1_weather[(plant1_weather['HOUR'] >= 20) | (plant1_weather['HOUR'] <= 5)]
night_irradiation_positive = night_irradiation[night_irradiation['IRRADIATION'] > 0]
print(f"\nNight hours (20:00-05:00) with positive irradiation: {len(night_irradiation_positive)} records")

# Check for reasonable temperature differences
temp_diff = plant1_weather['MODULE_TEMPERATURE'] - plant1_weather['AMBIENT_TEMPERATURE']
extreme_diff = plant1_weather[(temp_diff > 30) | (temp_diff < -10)]
print(f"\nExtreme temperature differences (>30°C or <-10°C): {len(extreme_diff)} records")
if len(extreme_diff) > 0:
    print(f"Temperature difference range: {temp_diff.min():.2f} to {temp_diff.max():.2f}")

# Clean up temporary columns
plant1_weather.drop(['DATE_TIME_DT', 'HOUR'], axis=1, inplace=True, errors='ignore')

=== DATA CONSISTENCY CHECKS ===

MODULE_TEMPERATURE < AMBIENT_TEMPERATURE: 1673 records
Sample records where module is colder than ambient:
             DATE_TIME  AMBIENT_TEMPERATURE  MODULE_TEMPERATURE
0  2020-05-15 00:00:00            25.184316           22.857507
1  2020-05-15 00:15:00            25.084589           22.761668
2  2020-05-15 00:30:00            24.935753           22.592306
3  2020-05-15 00:45:00            24.846130           22.360852
4  2020-05-15 01:00:00            24.621525           22.165423

Night hours (20:00-05:00) with positive irradiation: 29 records

Extreme temperature differences (>30°C or <-10°C): 23 records
Temperature difference range: -3.42 to 35.24


#### 3.5. Check for data type consistency

In [9]:
# Check data type consistency
print("=== DATA TYPE CONSISTENCY CHECKS ===")

# Expected types based on constraints
expected_types = {
    'DATE_TIME': 'object',  # Will be converted to datetime
    'PLANT_ID': 'int64',
    'SOURCE_KEY': 'object',
    'AMBIENT_TEMPERATURE': 'float64',
    'MODULE_TEMPERATURE': 'float64',
    'IRRADIATION': 'float64'
}

print("\nCurrent data types:")
print(plant1_weather.dtypes)

print("\nData type validation:")
type_issues = []
for col, expected_type in expected_types.items():
    actual_type = str(plant1_weather[col].dtype)
    if actual_type != expected_type:
        type_issues.append(f"{col}: expected {expected_type}, got {actual_type}")
        print(f"  ❌ {col}: expected {expected_type}, got {actual_type}")
    else:
        print(f"  ✅ {col}: {actual_type}")

if type_issues:
    print(f"\n⚠️  Found {len(type_issues)} data type issues")
else:
    print("\n✅ All data types are as expected")

# Test DATE_TIME conversion
print("\nTesting DATE_TIME format conversion...")
try:
    test_dates = pd.to_datetime(plant1_weather['DATE_TIME'].head(10), format='%Y-%m-%d %H:%M:%S')
    print("✅ DATE_TIME format is valid")
    print(f"Sample converted dates: {test_dates.tolist()[:3]}")
except Exception as e:
    print(f"❌ DATE_TIME format error: {e}")

=== DATA TYPE CONSISTENCY CHECKS ===

Current data types:
DATE_TIME               object
PLANT_ID                 int64
SOURCE_KEY              object
AMBIENT_TEMPERATURE    float64
MODULE_TEMPERATURE     float64
IRRADIATION            float64
dtype: object

Data type validation:
  ✅ DATE_TIME: object
  ✅ PLANT_ID: int64
  ✅ SOURCE_KEY: object
  ✅ AMBIENT_TEMPERATURE: float64
  ✅ MODULE_TEMPERATURE: float64
  ✅ IRRADIATION: float64

✅ All data types are as expected

Testing DATE_TIME format conversion...
✅ DATE_TIME format is valid
Sample converted dates: [Timestamp('2020-05-15 00:00:00'), Timestamp('2020-05-15 00:15:00'), Timestamp('2020-05-15 00:30:00')]


#### 3.6. Check for plausibility of data


In [10]:
# Check data plausibility
print("=== DATA PLAUSIBILITY CHECKS ===")

# Check against defined constraints
constraint_violations = []

# 1. Check PLANT_ID values
invalid_plant_ids = plant1_weather[~plant1_weather['PLANT_ID'].isin(CONSTRAINTS['PLANT_ID']['allowed_values'])]
if len(invalid_plant_ids) > 0:
    constraint_violations.append(f"PLANT_ID: {len(invalid_plant_ids)} invalid values")

# 2. Check numeric ranges
for col in ['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']:
    min_val = CONSTRAINTS[col]['min']
    max_val = CONSTRAINTS[col]['max']

    below_min = plant1_weather[plant1_weather[col] < min_val]
    above_max = plant1_weather[plant1_weather[col] > max_val]

    if len(below_min) > 0:
        constraint_violations.append(f"{col}: {len(below_min)} values below minimum ({min_val})")

    if len(above_max) > 0:
        constraint_violations.append(f"{col}: {len(above_max)} values above maximum ({max_val})")

# 3. Check date range
try:
    dates = pd.to_datetime(plant1_weather['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')
    min_date = pd.to_datetime(CONSTRAINTS['DATE_TIME']['range'][0], format='%Y-%m-%d %H:%M:%S')
    max_date = pd.to_datetime(CONSTRAINTS['DATE_TIME']['range'][1], format='%Y-%m-%d %H:%M:%S')

    invalid_dates = dates[(dates < min_date) | (dates > max_date)]
    if len(invalid_dates) > 0:
        constraint_violations.append(f"DATE_TIME: {len(invalid_dates)} values outside expected range")
except Exception as e:
    constraint_violations.append(f"DATE_TIME: Format conversion error - {e}")

# 4. Business logic checks
# Check for physically impossible temperature values
extreme_ambient = plant1_weather[(plant1_weather['AMBIENT_TEMPERATURE'] < -40) | (plant1_weather['AMBIENT_TEMPERATURE'] > 50)]
extreme_module = plant1_weather[(plant1_weather['MODULE_TEMPERATURE'] < -40) | (plant1_weather['MODULE_TEMPERATURE'] > 80)]

if len(extreme_ambient) > 0:
    constraint_violations.append(f"AMBIENT_TEMPERATURE: {len(extreme_ambient)} extreme values")

if len(extreme_module) > 0:
    constraint_violations.append(f"MODULE_TEMPERATURE: {len(extreme_module)} extreme values")

# 5. Check for unusual irradiation patterns
# Irradiation should be 0 during night hours and positive during day
plant1_weather['DATE_TIME_DT'] = pd.to_datetime(plant1_weather['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')
plant1_weather['HOUR'] = plant1_weather['DATE_TIME_DT'].dt.hour

# Check daytime (10:00-15:00) for zero irradiation
daytime_zero_irradiation = plant1_weather[(plant1_weather['HOUR'].between(10, 15)) & (plant1_weather['IRRADIATION'] == 0)]
if len(daytime_zero_irradiation) > 0:
    constraint_violations.append(f"IRRADIATION: {len(daytime_zero_irradiation)} zero values during peak daylight hours")

# Check for maximum irradiation values exceeding typical solar panel limits
max_irradiation = plant1_weather[plant1_weather['IRRADIATION'] > 1.2]
if len(max_irradiation) > 0:
    constraint_violations.append(f"IRRADIATION: {len(max_irradiation)} values exceeding typical maximum (1.2 kW/m²)")

print(f"\nFound {len(constraint_violations)} constraint violations:")
for violation in constraint_violations:
    print(f"  ⚠️  {violation}")

if len(constraint_violations) == 0:
    print("\n✅ All constraint checks passed!")
else:
    print(f"\n❌ Data quality issues detected. Review required.")

# Clean up temporary columns
plant1_weather.drop(['DATE_TIME_DT', 'HOUR'], axis=1, inplace=True, errors='ignore')

=== DATA PLAUSIBILITY CHECKS ===

Found 1 constraint violations:
  ⚠️  IRRADIATION: 1 values exceeding typical maximum (1.2 kW/m²)

❌ Data quality issues detected. Review required.
